**Grid-based estimation of $\pi$ - comparing convergence with the PRNG approach.**
The previous notebook (`mc_circle_prng.ipynb`) scattered random points over the
$[-1,1]\times[-1,1]$ square and counted how many landed inside the unit circle.
This notebook uses the same sample count but places the points on a
**uniform grid** instead.

The formula is identical:

$$\hat{\pi} = 4 \times \frac{\text{grid points inside circle}}{\text{grid points total}}$$

but the convergence behavior is fundamentally different:

| Method | Error scales as | Error at $N=102{,}400$ |
|---|---|---|
| Random (PRNG) | $O(1/\sqrt{N})$ | $\approx 0.09\%$ |
| Uniform grid | $O(1/N)$ | $\approx 0.03\%$ |

The grid is essentially a **midpoint Riemann sum** for the area integral.
Its error shrinks proportional to $1/N$ (like numerical integration),
while random sampling only improves as $1/\sqrt{N}$.
To halve the random error you need four times as many points;
to halve the grid error you only need twice as many.

---
**Defining the sampling region.**
Same bounding box as the PRNG notebook: the $2\times 2$ square
enclosing the unit circle, with `side_dots` grid points per side
for a total of `side_dots`$^2$ sample points.

In [ ]:
"""mc_circle_grid.ipynb"""

# Cell 01 - Define the sampling region and grid resolution

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, Rectangle

bbox = Rectangle((-1, -1), 2, 2).get_bbox()  # [-1,1] x [-1,1] square
print(bbox)

side_dots = 320
total_dots = side_dots**2  # 102,400 - same as PRNG notebook
print(f"{total_dots = :,}")

**Building the uniform grid.**
Rather than placing grid points at the boundary edges, the points are
**centered within each cell** by offsetting the linspace by half a cell width
($1/\text{side\_dots}$).
This is the midpoint rule: each point represents the center of a small
square cell of area $(2/\text{side\_dots})^2$.

`np.meshgrid` then produces all $320 \times 320$ pairings of $x$ and $y$
values, and `.flatten()` collapses them into two 1-D arrays that can be
processed the same way as the random points in the PRNG notebook.

In [ ]:
# Cell 02 - Build the uniform grid with cell-centered points
# Points are offset by half a cell width to implement the midpoint rule

half_step = 1 / side_dots  # half the cell width = 1/320
x = np.linspace(-1 + half_step, 1 - half_step, side_dots)
y = np.linspace(-1 + half_step, 1 - half_step, side_dots)

xv, yv = np.meshgrid(x, y)  # all (x, y) pairings: shape (320, 320)
x = xv.flatten()  # flatten to 1-D for uniform processing
y = yv.flatten()

print(f"x range : {x.min():.6f} to {x.max():.6f}  (cell-centered, not at boundary)")
print(f"cell width : {2 / side_dots:.6f}")
pd.DataFrame({"x": x[:5], "y": y[:5]})

**Distance from the origin - identical to the PRNG approach.**
The same vectorized computation applies regardless of how the points were generated.

In [ ]:
# Cell 03 - Euclidean distance from the origin for every grid point

d = np.sqrt(x**2 + y**2)

pd.DataFrame({"x": x[:5], "y": y[:5], "d": d[:5]})

**Classifying grid points as inside or outside the circle.**
Same boolean indexing as the PRNG notebook.

In [ ]:
# Cell 04 - Partition grid points: inside (d <= 1) vs outside (d > 1)

x_in = x[d <= 1.0]
y_in = y[d <= 1.0]
x_out = x[d > 1.0]
y_out = y[d > 1.0]

pd.DataFrame(
    {"x_in": x_in[:5], "y_in": y_in[:5], "x_out": x_out[:5], "y_out": y_out[:5]}
)

**Scatter plot of the uniform grid.**
Unlike the PRNG version, the dots form a perfectly regular pattern.
The boundary between red and blue follows the circle closely
but with visible staircase edges where grid cells straddle the circumference -
this staircase error is the main source of inaccuracy in the grid method.

In [ ]:
# Cell 05 - Scatter plot: red = inside, blue = outside, black circle = exact boundary

fig, ax = plt.subplots(figsize=(7, 7))
fig.canvas.manager.set_window_title("mc_circle_grid")
ax.scatter(x_in, y_in, color="red", marker=".", s=0.5)
ax.scatter(x_out, y_out, color="blue", marker=".", s=0.5)
ax.add_patch(Circle((0, 0), 1, fill=False, color="black", linewidth=1.5))
ax.set_aspect("equal")
ax.set_title(f"Uniform Grid Method ({total_dots:,} points)")
ax.set_xlabel("x")
ax.set_ylabel("y")
plt.tight_layout()
plt.show()

**Estimating $\pi$ and comparing with the PRNG result.**
With the same 102,400 points the grid method should produce a smaller
absolute error than the PRNG method,
reflecting its $O(1/N)$ vs $O(1/\sqrt{N})$ convergence advantage.

In [ ]:
# Cell 06 - Compute the pi estimate and compare with PRNG error

act = np.pi
est = (bbox.width * bbox.height) * np.count_nonzero(d <= 1.0) / total_dots
err = np.abs((est - act) / act)

print(f"dots total  : {total_dots:,}")
print(f"dots inside : {np.count_nonzero(d <= 1.0):,}")
print(f"actual pi   : {act:.6f}")
print(f"estimated   : {est:.6f}")
print(f"abs error   : {err:.5%}")
print()
print("PRNG error (same N, seed=2020) : 0.09230%")
print(f"Grid error                     : {err:.5%}")
print(f"Grid is ~{0.09230 / (err * 100):.1f}x more accurate for the same sample count")

**Why use Monte Carlo if the grid method is more accurate in 2D?**
In two dimensions the midpoint grid is clearly superior - same number of points,
smaller error, predictable convergence.
The grid's advantage comes from the **curse of dimensionality**, and it works
against itself in higher dimensions.

A uniform grid with $k$ points per axis requires $k^d$ total points in $d$ dimensions.
To achieve 1% accuracy in 2D you might need $k^2 = 10{,}000$ points.
In 10 dimensions that same grid needs $k^{10} = 10^{20}$ points -
more than the number of atoms in the human body.
The grid method simply becomes computationally impossible as dimension grows.

Monte Carlo's $O(1/\sqrt{N})$ convergence rate is **independent of dimension**.
The same 10,000 points give roughly the same accuracy whether the integration
domain is 2D or 200D.
This is Monte Carlo's defining advantage.

In a later lab we will use Monte Carlo to compute the volume of $n$-dimensional
unit balls for $n = 1$ through $12$.
A grid approach would be completely intractable for $n \geq 6$ or so,
while Monte Carlo handles all twelve dimensions with the same code and
the same sample count.
The 2D circle notebooks are therefore a warm-up: they introduce the method
in a setting where you can verify the answer easily ($\pi$ is well known)
before applying it to problems where no grid alternative exists.